# Named entity recognition

Run the cells in order.

## Words and tags

In named entity recognition we assign a tag to each word so that a sequence such as “Ada Lovelace” can be recognised as one person, using the IOB2 convention in which B-PER starts a person entity, I-PER continues it, and O marks a word outside an entity.

In [ ]:
import json
import torch
from torch import nn
from torch.nn.utils.rnn import pad_sequence

torch.manual_seed(7)
torch.set_num_threads(1)
examples = [
    ("Ada Lovelace visits London", ["B-PER", "I-PER", "O", "B-LOC"]),
    ("Acme hired Ada", ["B-ORG", "O", "B-PER"]),
]
labels = "O B-PER I-PER B-ORG I-ORG B-LOC I-LOC B-MISC I-MISC".split()
for text, tags in examples:
    print(list(zip(text.split(), tags)))

**Question:** If Lovelace changes from I-PER to B-PER, how many person entities are annotated?

## Word IDs and padding

To process sentences of different lengths together, we turn their words and tags into integer IDs and pad the shorter sentence until both rows have the same length, using 0 for missing input words and -100 for missing targets so that the loss can distinguish padding from an actual label.

In [ ]:
vocab = {"<PAD>": 0, "<UNK>": 1}
for text, _ in examples:
    for word in text.lower().split():
        vocab.setdefault(word, len(vocab))

ids = [torch.tensor([vocab[w] for w in text.lower().split()])
       for text, _ in examples]
gold = [torch.tensor([labels.index(tag) for tag in tags])
        for _, tags in examples]
x = pad_sequence(ids, batch_first=True, padding_value=0)
y = pad_sequence(gold, batch_first=True, padding_value=-100)
print("Word IDs:", x, "\nTargets:", y)
print("Shapes:", x.shape, y.shape)

**Question:** What happens if we use 0 to pad the targets as well?

## Embedding lookup

Passing our word IDs through nn.Embedding replaces each integer with the eight learned values stored in its row of the embedding table, turning an input tensor shaped (batch, words) into a tensor shaped (batch, words, features) while preserving which vector belongs to each sentence and word position.

In [ ]:
embedding = nn.Embedding(len(vocab), 8, padding_idx=0)
vectors = embedding(x)
print("IDs:", x.shape)
print("Vectors:", vectors.shape)
print("Ada has the same vector in both sentences:",
      torch.equal(vectors[0, 0], vectors[1, 2]))

**Question:** What changes in the output shape if embedding width increases from 8 to 16?

## A score for every word and label

We apply the hidden linear layer, ReLU and output layer to each word vector while preserving the batch and word axes, so that the final tensor contains nine raw label scores for every word position, including padded positions whose scores will be ignored when we calculate the loss.

In [ ]:
class TokenTagger(nn.Module):
    def __init__(self, embedding):
        super().__init__()
        self.embedding = embedding
        self.hidden = nn.Linear(8, 16)
        self.relu = nn.ReLU()
        self.output = nn.Linear(16, len(labels))

    def forward(self, ids):
        features = self.embedding(ids)
        hidden = self.relu(self.hidden(features))
        return self.output(hidden)

model = TokenTagger(embedding)
scores = model(x)
print("Scores:", scores.shape)

**Question:** If we average features over the word axis, what happens to the output shape?

### Alternative A

If we average the word vectors before applying the hidden and output layers, will the result still contain one prediction for every word, and which tensor axis would disappear?

In [ ]:
with torch.no_grad():
    alt_vectors = model.embedding(x).mean(dim=1)
    alt_scores = model.output(model.relu(model.hidden(alt_vectors)))
print("Alternative scores:", tuple(alt_scores.shape))
print("Working scores:", tuple(model(x).shape))

## Choose one label per word

Because the last axis of our score tensor contains the nine possible labels, argmax(dim=-1) compares those scores separately for each word and returns the winning label index, changing the shape from (2, 4, 9) to (2, 4) while keeping each prediction attached to its original sentence and word position.

In [ ]:
scores = model(x)
prediction = scores.argmax(dim=-1)
print("Scores:", scores.shape)
print("Predicted IDs:", prediction.shape)
print(prediction)

**Question:** If we change argmax to dim=0, which axis will it compare and what will the resulting tensor contain?

### Alternative A

If argmax compares dim=0 instead of the last axis, what will each returned number identify, and does the resulting shape still provide one label index per word?

In [ ]:
with torch.no_grad():
    alt_scores = model(x)
    alt_predictions = alt_scores.argmax(dim=0)
print("Alternative shape:", tuple(alt_predictions.shape))
print("Alternative indices:", alt_predictions)
print("Largest index:", alt_predictions.max().item())

## Cross-entropy and padding

Our model produces scores with shape (2, 4, 9), so we reshape them into eight rows of nine label scores and flatten the targets in the same order, allowing cross-entropy to compare each word’s scores with its correct label while ignore_index=-100 excludes the padded position from the loss.

In [ ]:
criterion = nn.CrossEntropyLoss(ignore_index=-100)
scores = model(x)
flat_scores = scores.reshape(-1, len(labels))
flat_targets = y.reshape(-1)
loss = criterion(flat_scores, flat_targets)
print("Before:", scores.shape, y.shape)
print("After:", flat_scores.shape, flat_targets.shape)
print("Positions used:", (flat_targets != -100).sum().item())
print("Loss:", round(loss.item(), 4))

**Question:** What error appears if criterion receives scores and y without reshaping?

### Alternative A

If the target tensor still has shape (batch, words), can CrossEntropyLoss accept the model’s original (batch, words, labels) scores directly, or does it expect the class axis somewhere else?

In [ ]:
with torch.no_grad():
    alt_scores = model(x)
try:
    alt_loss = criterion(alt_scores, y)
    print("Alternative loss:", alt_loss.item())
except RuntimeError as error:
    print(type(error).__name__ + ": " + str(error))

### Alternative B

If we move the label axis to dim=1 instead of flattening the word positions, will cross-entropy compare the same scores and targets, and should the two scalar losses agree?

In [ ]:
with torch.no_grad():
    alt_scores = model(x)
    alt_flat_loss = criterion(alt_scores.reshape(-1, len(labels)), y.reshape(-1))
    alt_transposed_loss = criterion(alt_scores.transpose(1, 2), y)
print("Losses agree:", torch.allclose(alt_flat_loss, alt_transposed_loss, atol=1e-6))
print("Transposed scores:", tuple(alt_scores.transpose(1, 2).shape))

### Alternative C

If we replace the ignored target -100 with the real label ID zero, will the loss still use only the seven actual words, or will it start learning from an additional position?

In [ ]:
alt_targets = y.masked_fill(y == -100, 0)
print("Working positions scored:", (y != -100).sum().item())
print("Alternative positions scored:", (alt_targets != -100).sum().item())
print("Extra targets:", alt_targets[y == -100].tolist())
print("Meaning of target zero:", labels[0])

## Train on the two sentences

During each training iteration we clear the previous gradients, calculate a new loss from the current scores, use backward() to calculate how the parameters affect that loss, and call optimizer.step() to update the weights, repeating this process on our two sentences to check whether the model can fit their labels.

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=0.03)
model.train()
loss_history = []
for epoch in range(80):
    optimizer.zero_grad()
    scores = model(x)
    loss = criterion(scores.reshape(-1, len(labels)), y.reshape(-1))
    loss.backward()
    optimizer.step()
    loss_history.append(loss.item())
print("Loss:", round(loss_history[0], 4), "->", round(loss_history[-1], 4))

**Question:** What happens to the loss if we remove optimizer.step() and rerun from a fresh model?

## Read the predicted tags

After training, we calculate label scores with gradient recording disabled and convert them into predicted tags, keeping only the original number of words in each sentence so that padding added for batching is excluded from the word-and-tag pairs we display.

In [ ]:
model.eval()
with torch.no_grad():
    scores = model(x)
    prediction = scores.argmax(dim=-1)
predicted_tags = []
for row, (text, tags) in zip(prediction, examples):
    predicted = [labels[i] for i in row[:len(tags)].tolist()]
    predicted_tags.append(predicted)
    print(list(zip(text.split(), predicted)))

**Question:** Why can the padded position have a predicted label even though its embedding is zero?

## Score complete entities

Entity evaluation checks whether a predicted span has the same start, end and type as a gold entity, so predicting only “Ada” as a person gives an incorrect person span even though that word’s tag is correct, while predicting “London” as a location gives one complete entity match.

In [ ]:
from seqeval.metrics import accuracy_score, precision_score, recall_score, f1_score
from seqeval.scheme import IOB2

gold_tags = [["B-PER", "I-PER", "O", "B-LOC"]]
guessed_tags = [["B-PER", "O", "O", "B-LOC"]]
metric_args = dict(mode="strict", scheme=IOB2, zero_division=0)
print("Token accuracy:", accuracy_score(gold_tags, guessed_tags))
print("Entity precision:", precision_score(gold_tags, guessed_tags, **metric_args))
print("Entity recall:", recall_score(gold_tags, guessed_tags, **metric_args))
print("Entity F1:", f1_score(gold_tags, guessed_tags, **metric_args))
training_gold = [tags for _, tags in examples]
print("Training F1:", f1_score(training_gold, predicted_tags, **metric_args))

**Question:** If we repair the person boundary, what happens to token accuracy and entity F1?

## Change a neighbouring word

To find out whether our classifier uses context, we keep Ada’s word ID fixed and change the following word from visits to hired, then compare Ada’s nine label scores across the two inputs to see whether information from the changed neighbour can reach its prediction.

In [ ]:
a = torch.tensor([[vocab["ada"], vocab["visits"]]])
b = torch.tensor([[vocab["ada"], vocab["hired"]]])
model.eval()
with torch.no_grad():
    ada_a = model(a)[0, 0]
    ada_b = model(b)[0, 0]
print("Same Ada scores:", torch.equal(ada_a, ada_b))

**Question:** Would longer training let this architecture use the neighbouring word?

## Add a bidirectional LSTM

A bidirectional LSTM processes the word vectors in both directions and concatenates its two eight-feature states at each position, giving the output layer sixteen features per word through which information from neighbouring words can affect the label scores, as we demonstrate using two sentences of equal length.

In [ ]:
class ContextTagger(nn.Module):
    def __init__(self):
        super().__init__()
        self.embedding = nn.Embedding(len(vocab), 8, padding_idx=0)
        self.lstm = nn.LSTM(8, 8, batch_first=True, bidirectional=True)
        self.output = nn.Linear(16, len(labels))

    def forward(self, ids):
        vectors = self.embedding(ids)
        states, _ = self.lstm(vectors)
        return self.output(states)

torch.manual_seed(9)
contextual = ContextTagger().eval()
with torch.no_grad():
    context_scores = contextual(torch.cat([a, b]))
print("Scores:", context_scores.shape)
print("Same Ada scores:", torch.equal(context_scores[0, 0], context_scores[1, 0]))

**Question:** Which direction carries information from visits or hired back to Ada?